<a href="https://colab.research.google.com/github/jayasinghesasmitha/EN4554-Deep-Learning-For-Vision/blob/CNN-assignment-2/220261V_a02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EN4554 Assignment 2 – CNNs (PyTorch)

**Name:** Jayasinghe J.A.S.U.  **Index No.:** 220261V
**GitHub:** https://github.com/jayasinghesasmitha/EN4554-Deep-Learning-For-Vision.git

**Settings used throughout:** CIFAR-10, 45k train / 5k validation split (fixed seed), per-channel normalisation with *training* statistics, seed = 42. The test set is touched exactly once, in Task 4.

In [1]:
import math, time, random, copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_type = 'cuda' if device == 'cuda' else 'cpu'
USE_AMP = (device == 'cuda')
QUICK = False
E = lambda n: 1 if QUICK else n
print('device:', device, '| AMP:', USE_AMP, '| torch', torch.__version__)

device: cuda | AMP: True | torch 2.11.0+cu130


## Data: CIFAR-10, 45k/5k split, normalisation with training statistics

In [2]:
tv_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
Xall = torch.from_numpy(tv_train.data).permute(0, 3, 1, 2).float() / 255.0
yall = torch.tensor(tv_train.targets)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(50000, generator=g)
tr_idx, va_idx = perm[:45000], perm[45000:]
Xtr, ytr, Xva, yva = Xall[tr_idx], yall[tr_idx], Xall[va_idx], yall[va_idx]

mean = Xtr.mean(dim=(0, 2, 3), keepdim=True)
std  = Xtr.std(dim=(0, 2, 3), keepdim=True)
Xtr, Xva = (Xtr - mean) / std, (Xva - mean) / std
Xtr, ytr, Xva, yva = Xtr.to(device), ytr.to(device), Xva.to(device), yva.to(device)
DATA = (Xtr, ytr, Xva, yva)
print(Xtr.shape, Xva.shape, 'mean', mean.flatten().tolist(), 'std', std.flatten().tolist())

100%|██████████| 170M/170M [47:56<00:00, 59.3kB/s]


torch.Size([45000, 3, 32, 32]) torch.Size([5000, 3, 32, 32]) mean [0.49165481328964233, 0.4823332130908966, 0.4466792047023773] std [0.24708116054534912, 0.2435086965560913, 0.2615806758403778]


## Shared helpers (batching, augmentation, cutout, mixup, schedule, train/eval loops)

In [3]:
def batches(X, y, bs, shuffle=True):
    n = X.shape[0]
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for i in range(0, n, bs):
        j = idx[i:i + bs]
        if len(j) < 2: continue
        yield X[j], y[j]

def flip_crop(x, pad=4):
    """random horizontal flip + random 32x32 crop from the zero-padded (pad=4) image"""
    n, c, h, w = x.shape; d = x.device
    flip = torch.rand(n, device=d) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    xp = F.pad(x, (pad, pad, pad, pad))
    i = torch.randint(0, 2 * pad + 1, (n,), device=d)
    j = torch.randint(0, 2 * pad + 1, (n,), device=d)
    rows = i[:, None] + torch.arange(h, device=d)[None]
    cols = j[:, None] + torch.arange(w, device=d)[None]
    return xp[torch.arange(n, device=d)[:, None, None, None],
              torch.arange(c, device=d)[None, :, None, None],
              rows[:, None, :, None], cols[:, None, None, :]]

def cutout(x, size=8):
    """zero out one random size x size square per image (centre may lie near the border, square is clipped)"""
    n, c, h, w = x.shape; d = x.device
    cy = torch.randint(0, h, (n,), device=d)[:, None, None]
    cx = torch.randint(0, w, (n,), device=d)[:, None, None]
    ys = torch.arange(h, device=d)[None, :, None]
    xs = torch.arange(w, device=d)[None, None, :]
    hole = (ys >= cy - size // 2) & (ys < cy + size // 2) & (xs >= cx - size // 2) & (xs < cx + size // 2)
    return x * (~hole)[:, None].to(x.dtype)

def mixup_batch(x, y, alpha=0.4):
    """returns mixed inputs, the two label sets and lambda; loss = lam*CE(ya) + (1-lam)*CE(yb)"""
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def make_sched(opt, total_steps, warmup=0):
    """per-iteration linear warm-up then cosine decay"""
    def f(s):
        if s < warmup: return (s + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total_steps - warmup)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def evaluate(model, X, y, bs=1000):
    model.eval(); tot, correct = 0.0, 0
    with torch.no_grad(), torch.autocast(dev_type, enabled=USE_AMP):
        for i in range(0, len(X), bs):
            out = model(X[i:i + bs]).float()
            tot += F.cross_entropy(out, y[i:i + bs], reduction='sum').item()
            correct += (out.argmax(1) == y[i:i + bs]).sum().item()
    return tot / len(X), correct / len(X)

def n_params(m): return sum(p.numel() for p in m.parameters())

def train_model(model, opt, epochs, data, bs=128, aug=(), mixup=0.0, ls=0.0,
                sched=None, eval_n=5000, name='', verbose=True):
    """train_loss = running mean of the (augmented) training loss during the epoch;
       train_acc  = accuracy in eval mode on the first eval_n CLEAN training images;
       val_*      = metrics on the 5k validation split."""
    Xtr, ytr, Xva, yva = data
    scaler = torch.amp.GradScaler(dev_type, enabled=USE_AMP)
    h = {k: [] for k in ['train_loss', 'train_acc', 'val_loss', 'val_acc']}
    for ep in range(epochs):
        model.train(); t0 = time.time()
        run = torch.zeros((), device=Xtr.device); seen = 0
        for xb, yb in batches(Xtr, ytr, bs):
            for f in aug: xb = f(xb)
            with torch.autocast(dev_type, enabled=USE_AMP):
                if mixup > 0:
                    xb, ya, yb2, lam = mixup_batch(xb, yb, mixup)
                    out = model(xb)
                    loss = lam * F.cross_entropy(out, ya, label_smoothing=ls) + \
                           (1 - lam) * F.cross_entropy(out, yb2, label_smoothing=ls)
                else:
                    out = model(xb)
                    loss = F.cross_entropy(out, yb, label_smoothing=ls)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
            run += loss.detach().float() * len(xb); seen += len(xb)
        tl = (run / seen).item()
        if not math.isfinite(tl):
            for _ in range(ep, epochs):
                h['train_loss'].append(float('nan')); h['val_loss'].append(float('nan'))
                h['train_acc'].append(0.1); h['val_acc'].append(0.1)
            print(f'[{name}] diverged at epoch {ep+1}'); break
        _, ta = evaluate(model, Xtr[:eval_n], ytr[:eval_n]); vl, va = evaluate(model, Xva, yva)
        h['train_loss'].append(tl); h['train_acc'].append(ta); h['val_loss'].append(vl); h['val_acc'].append(va)
        if verbose:
            print(f'[{name}] ep {ep+1:3d}/{epochs} loss {tl:.4f} train_acc {ta:.4f} val_loss {vl:.4f} val_acc {va:.4f} ({time.time()-t0:.1f}s)')
    return h

class SmallCNN(nn.Module):
    """3 blocks (conv[-BN]-ReLU) x reps + maxpool, then FC. Used in Tasks 2c and 3a."""
    def __init__(self, bn=True, drop=0.0, reps=2, widths=(32, 64, 128)):
        super().__init__()
        layers, c = [], 3
        for w in widths:
            for _ in range(reps):
                layers.append(nn.Conv2d(c, w, 3, padding=1, bias=not bn))
                if bn: layers.append(nn.BatchNorm2d(w))
                layers.append(nn.ReLU(inplace=True)); c = w
            layers.append(nn.MaxPool2d(2))
            if drop > 0: layers.append(nn.Dropout(drop / 2))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(c * 16, 256), nn.ReLU(inplace=True),
                                        nn.Dropout(drop), nn.Linear(256, 10))
    def forward(self, x): return self.classifier(self.features(x))

xs = Xtr[:4]
assert flip_crop(xs).shape == xs.shape and cutout(xs).shape == xs.shape
print('helpers OK; zeroed fraction by cutout:', (cutout(xs) == 0).float().mean().item())

helpers OK; zeroed fraction by cutout: 0.056640625
